In [45]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

from tqdm import tqdm

from medmnist import BreastMNIST
from medmnist import INFO

In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Using: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("CUDA is not available. Using CPU.")

Using: NVIDIA GeForce RTX 4070 Ti SUPER
CUDA: 12.4


In [47]:
info = INFO['breastmnist']
data_flag = 'breastmnist'
DataClass = BreastMNIST

task = info['task']  
n_channels = info['n_channels']
n_classes = len(info['label'])

print(f"Number of classes:", n_classes)
print(f"Number of channels: {n_channels}")

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),  
    transforms.RandomRotation(degrees=15),  
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

Number of classes: 2
Number of channels: 1


In [48]:
data_train = DataClass(split='train', transform=train_transform, download=True, size=28)
data_test = DataClass(split='test', transform=eval_transform, download=True, size=28)
data_eval = DataClass(split='val', transform=eval_transform, download=True, size=28)

Using downloaded and verified file: /home/eflammere/.medmnist/breastmnist.npz
Using downloaded and verified file: /home/eflammere/.medmnist/breastmnist.npz
Using downloaded and verified file: /home/eflammere/.medmnist/breastmnist.npz


In [49]:
batch_size = 128

dataloader_train = DataLoader(dataset=data_train, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(dataset=data_test, batch_size=batch_size, shuffle=False)
dataloader_eval = DataLoader(dataset=data_eval, batch_size=batch_size, shuffle=False)

print(f"\nNumber of images in training dataset: {len(data_train)}")
print(f"Number of images in test dataset: {len(data_test)}")
print(f"Number of images in validation dataset: {len(data_eval)}")



Number of images in training dataset: 546
Number of images in test dataset: 156
Number of images in validation dataset: 78


In [50]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.fc = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 8)
)

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.1)
criterion = nn.CrossEntropyLoss().to(device)
epochs = 100

In [51]:
train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []
val_precisions, val_recalls, val_f1_scores, val_aucs = [], [], [], []

best_val_acc = 0.0 
best_model_path = "/home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/notebooks/classical/best_model.pth"

for epoch in range(epochs):
    print(f"\nEpoch: {epoch+1}/{epochs}")

    model.train()
    total_loss, total_acc = 0.0, 0.0

    for batch_idx, (inputs, labels) in enumerate(dataloader_train):
        inputs, labels = inputs.to(device), labels.squeeze().to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels.long())
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        batch_acc = (predicted == labels).float().mean().item()
        total_acc += batch_acc

        print(f"Batch {batch_idx+1}/{len(dataloader_train)}\n Loss: {loss.item():.4f} | Acc: {batch_acc:.4f}")

    train_losses.append(total_loss / len(dataloader_train))
    train_accuracies.append(total_acc / len(dataloader_train))

    print(f"\nEpoch {epoch+1} Summary\n Train Loss: {train_losses[-1]:.4f} | Train Acc: {train_accuracies[-1]:.4f}")


    model.eval()
    val_loss, val_acc = 0.0, 0.0
    val_labels, val_predictions, val_probs = [], [], []

    with torch.no_grad():
        for inputs, labels in dataloader_eval:
            inputs, labels = inputs.to(device), labels.squeeze().to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels.long())
            val_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1) 
            
            val_labels.extend(labels.cpu().numpy())
            val_predictions.extend(preds.cpu().numpy())
            val_probs.extend(probs.cpu().numpy()) 

    val_losses.append(val_loss / len(dataloader_eval))
    val_accuracies.append(accuracy_score(val_labels, val_predictions))
    
    val_precision = precision_score(val_labels, val_predictions, average='macro', zero_division=0)
    val_recall = recall_score(val_labels, val_predictions, average='macro', zero_division=0)
    val_f1 = f1_score(val_labels, val_predictions, average='macro', zero_division=0)
    
    try:
        auc_val = roc_auc_score(val_labels, val_probs, multi_class='ovr')
    except ValueError:
        auc_val = 0.0 

    val_precisions.append(val_precision)
    val_recalls.append(val_recall)
    val_f1_scores.append(val_f1)
    val_aucs.append(auc_val)

    print(f"Val Loss: {val_losses[-1]:.4f} | Acc: {val_accuracies[-1]:.4f} | Precision: {val_precision*100:.4f} | Recall: {val_recall*100:.4f} | F1: {val_f1*100:.4f} | AUC: {auc_val*100:.4f}")

    if val_accuracies[-1] > best_val_acc:
        best_val_acc = val_accuracies[-1]
        torch.save(model.state_dict(), best_model_path)
        print(f"New best model saved with validation accuracy of {best_val_acc*100:.4f}")

print(f"Best model saved at {best_model_path} with validation accuracy of {best_val_acc*100:.4f}")



Epoch: 1/100
Batch 1/5
 Loss: 2.0864 | Acc: 0.0625
Batch 2/5
 Loss: 2.0746 | Acc: 0.0859
Batch 3/5
 Loss: 2.0710 | Acc: 0.1016
Batch 4/5
 Loss: 2.0696 | Acc: 0.0859
Batch 5/5
 Loss: 2.0189 | Acc: 0.1471

Epoch 1 Summary
 Train Loss: 2.0641 | Train Acc: 0.0966
Val Loss: 2.0367 | Acc: 0.3974 | Precision: 36.4706 | Recall: 12.6817 | F1: 17.3077 | AUC: 0.0000
New best model saved with validation accuracy of 39.7436

Epoch: 2/100
Batch 1/5
 Loss: 2.0699 | Acc: 0.0938
Batch 2/5
 Loss: 2.0481 | Acc: 0.1094
Batch 3/5
 Loss: 2.0705 | Acc: 0.1016
Batch 4/5
 Loss: 2.0294 | Acc: 0.1484
Batch 5/5
 Loss: 2.0220 | Acc: 0.1765

Epoch 2 Summary
 Train Loss: 2.0480 | Train Acc: 0.1259
Val Loss: 2.0356 | Acc: 0.3590 | Precision: 25.8824 | Recall: 10.4261 | F1: 13.6073 | AUC: 0.0000

Epoch: 3/100
Batch 1/5
 Loss: 2.0346 | Acc: 0.1797
Batch 2/5
 Loss: 1.9968 | Acc: 0.1875
Batch 3/5
 Loss: 2.0203 | Acc: 0.2188
Batch 4/5
 Loss: 2.0053 | Acc: 0.2578
Batch 5/5
 Loss: 2.0252 | Acc: 0.1765

Epoch 3 Summary
 Tra

In [58]:
model_path = "/home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/notebooks/classical/best_model.pth"
model.load_state_dict(torch.load(model_path, weights_only=True))

test_loss = 0.0
test_labels, test_predictions, test_probs = [], [], []

model.eval()
with torch.no_grad():
    for inputs, labels in dataloader_test:
        inputs = inputs.squeeze(1).to(device)
        labels = labels.squeeze().to(device).long()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1) 
        
        test_labels.extend(labels.cpu().numpy())
        test_predictions.extend(preds.cpu().numpy())
        test_probs.extend(probs.cpu().numpy())  

test_labels = np.array(test_labels)
test_predictions = np.array(test_predictions)
test_probs = np.array(test_probs)

test_accuracy = accuracy_score(test_labels, test_predictions)
test_precision = precision_score(
    test_labels, test_predictions, 
    average="weighted", zero_division=0
)
test_recall = recall_score(
    test_labels, test_predictions, 
    average="weighted", zero_division=0
)
test_f1 = f1_score(
    test_labels, test_predictions, 
    average="weighted", zero_division=0
)
test_auc = roc_auc_score(
    test_labels, test_probs[:, 1]
)

print(f"Test Loss: {test_loss / len(dataloader_test):.4f} | Acc: {test_accuracy:.4f} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f} | F1: {test_f1:.4f} | AUC: {test_auc:.4f}")


Test Loss: 0.4658 | Acc: 0.8333 | Precision: 0.8441 | Recall: 0.8333 | F1: 0.8111 | AUC: 0.8876


In [59]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model = nn.Sequential(*list(model.children())[:-1]) 

state_dict = torch.load(best_model_path, map_location=device, weights_only="True")

filtered_state_dict = {k: v for k, v in state_dict.items() if "fc" not in k}

model.load_state_dict(filtered_state_dict, strict=False)

model.to(device)
model.eval()

def extract_features(dataloader, dataset_name, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    features_list, labels_list = [], []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Extracting Features ({dataset_name})"):
            images = images.to(device)
            features = model(images) 

            features = features.view(features.size(0), -1)

            features_list.append(features.cpu().numpy())
            labels_list.append(labels.cpu().numpy())

    features_array = np.concatenate(features_list, axis=0)
    labels_array = np.concatenate(labels_list, axis=0)

    np.save(os.path.join(output_dir, f"features.npy"), features_array)
    np.save(os.path.join(output_dir, f"labels.npy"), labels_array)

    df = pd.DataFrame(features_array)
    df['label'] = labels_array
    df.to_csv(os.path.join(output_dir, f"features.csv"), index=False, header=False)

    print(f"{dataset_name} features saved in {output_dir}.")

output_directory = "/home/eflammere/HybridEmbeddingsForBreastCancerDetection/embeddings/classical"
extract_features(dataloader_train, "BreastMNIST", output_directory)

Extracting Features (BreastMNIST): 100%|██████████| 5/5 [00:00<00:00, 31.12it/s]


BreastMNIST features saved in /home/eflammere/HybridEmbeddingsForBreastCancerDetection/embeddings/classical.
